# Forward & Backward Selection — Hands-on Practice

`20_Feature_Selection_techniques.ipynb` covered Forward Selection and
Backward Elimination on `diabetes.csv` — but that notebook got built in
one big pass, so it reads more like a finished article than something
you built yourself.

This notebook redoes the same two techniques, from scratch, on a
**made-up dataset** — one line at a time. I'll explain each step, you
type it in and run it, we check the output together before moving on.

## Step 1 — Imports

`numpy` builds the random synthetic dataset next. The rest match
notebook 20/23: `train_test_split`, `DecisionTreeClassifier`,
`accuracy_score`, and `SequentialFeatureSelector` (aliased `SFS`) for
the search itself.

In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from mlxtend.feature_selection import SequentialFeatureSelector as SFS


## Step 2 — Build a made-up dataset, rigged on purpose

`Target` is 150 coin flips. `Feature1`/`Feature2` are built *from*
`Target` plus random noise — real signal, `Feature1` stronger than
`Feature2`. `Feature3`/`4`/`5` are pure random numbers with no
connection to `Target` at all — noise, on purpose. `rng =
np.random.default_rng(18)` fixes the seed so the numbers are the same
every time this runs.

In [27]:
rng = np.random.default_rng(18)
n = 150

Target = rng.integers(0, 2, n)

Feature1 = Target * 1.8 + rng.normal(0, 1.0, n)
Feature2 = Target * 1.1 + rng.normal(0, 1.0, n)
Feature3 = rng.normal(0, 1.0, n)
Feature4 = rng.normal(0, 1.0, n)
Feature5 = rng.normal(0, 1.0, n)

dataset = pd.DataFrame({
    "Feature1": Feature1, "Feature2": Feature2, "Feature3": Feature3,
    "Feature4": Feature4, "Feature5": Feature5, "Target": Target,
})
dataset.head()


,Feature1,Feature2,Feature3,Feature4,Feature5,Target
0,2.161596,1.335954,-0.403696,0.048703,-0.352747,1
1,-0.980579,0.071353,0.288733,-0.439909,-0.578918,0
2,0.567174,-0.811513,0.056646,-1.168114,0.306735,0
3,1.134049,-0.711547,-0.076631,0.726083,1.106567,1
4,0.631721,1.273348,-1.062992,-0.264813,-0.738309,1


## Step 3 — Look at the data

A shape check before doing anything else. Expect `(150, 6)`: 150
students, 5 feature columns plus `Target`.

In [28]:
dataset.shape


(150, 6)

Check the class balance before splitting — how many `1`s versus `0`s
actually came out of the 150 coin flips.

In [29]:
dataset["Target"].value_counts()


Target
1    85
0    65
Name: count, dtype: int64

## Step 4 — Split into X and y

`X` drops `Target` — only what the model is allowed to see. `y` is
`Target` alone, the answer key, kept separate.

In [31]:
X = dataset.drop(columns=["Target"])
y = dataset["Target"]


A look at both pieces side by side — same 150 rows, just divided by
column.

In [32]:
X, y

(     Feature1  Feature2  Feature3  Feature4  Feature5
 0    2.161596  1.335954 -0.403696  0.048703 -0.352747
 1   -0.980579  0.071353  0.288733 -0.439909 -0.578918
 2    0.567174 -0.811513  0.056646 -1.168114  0.306735
 3    1.134049 -0.711547 -0.076631  0.726083  1.106567
 4    0.631721  1.273348 -1.062992 -0.264813 -0.738309
 ..        ...       ...       ...       ...       ...
 145 -0.266569  0.954740 -1.347425 -0.104743 -0.776740
 146  1.218765  0.401340  0.393746  0.362350 -1.043485
 147  0.648704 -0.009586 -0.605813  1.161052 -1.784982
 148  1.590029 -0.770502 -0.132640 -1.196658  0.006370
 149  1.895690  0.019675  2.084451  0.551775 -0.504388
 
 [150 rows x 5 columns],
 0      1
 1      0
 2      0
 3      1
 4      1
       ..
 145    0
 146    1
 147    0
 148    0
 149    1
 Name: Target, Length: 150, dtype: int64)

## Step 5 — Split into train and test

Same call as notebook 20/23: `test_size=0.2` → 80/20 split,
`random_state=42` → reproducible shuffle, `stratify=y` → keep the
class mix similar in both groups instead of risking an unlucky
shuffle.

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((120, 5), (30, 5), (120,), (30,))

Proof, not just a claim, that `random_state` controls reproducibility:
run `train_test_split` twice with no seed (rows should differ), then
twice with `random_state=42` (rows should match exactly). A throwaway
check, not part of the main pipeline.

In [34]:
a = train_test_split(X, y, test_size=0.2)[0].index[:5]
b = train_test_split(X, y, test_size=0.2)[0].index[:5]
a, b   # no random_state — these will very likely differ

c = train_test_split(X, y, test_size=0.2, random_state=42)[0].index[:5]
d = train_test_split(X, y, test_size=0.2, random_state=42)[0].index[:5]
c, d   # random_state=42 — these will be identical


(Index([22, 15, 65, 11, 42], dtype='int64'),
 Index([22, 15, 65, 11, 42], dtype='int64'))

## Step 6 — Build the baseline model, train it

Same model as notebook 20/23: `DecisionTreeClassifier(max_depth=4,
random_state=42)`. `.fit()` trains on all 120 rows and **all 5
features, noise included** — this becomes the reference number every
later result gets compared against.

In [35]:
baseline_model = DecisionTreeClassifier(max_depth=4, random_state=42)
baseline_model.fit(X_train, y_train)


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at 

Look inside the trained tree — same `export_text()` trick as notebook
23, just on a deeper tree (`max_depth=4`) with 5 candidate features
instead of 1.

In [36]:
from sklearn.tree import export_text
print(export_text(baseline_model, feature_names=list(X_train.columns)))


|--- Feature1 <= 0.70
|   |--- Feature2 <= 0.39
|   |   |--- Feature4 <= -1.78
|   |   |   |--- Feature2 <= 0.07
|   |   |   |   |--- class: 1
|   |   |   |--- Feature2 >  0.07
|   |   |   |   |--- class: 0
|   |   |--- Feature4 >  -1.78
|   |   |   |--- class: 0
|   |--- Feature2 >  0.39
|   |   |--- Feature1 <= -0.23
|   |   |   |--- class: 0
|   |   |--- Feature1 >  -0.23
|   |   |   |--- Feature4 <= 1.36
|   |   |   |   |--- class: 1
|   |   |   |--- Feature4 >  1.36
|   |   |   |   |--- class: 0
|--- Feature1 >  0.70
|   |--- Feature2 <= -0.20
|   |   |--- Feature1 <= 1.74
|   |   |   |--- Feature5 <= -0.46
|   |   |   |   |--- class: 0
|   |   |   |--- Feature5 >  -0.46
|   |   |   |   |--- class: 0
|   |   |--- Feature1 >  1.74
|   |   |   |--- Feature1 <= 2.65
|   |   |   |   |--- class: 1
|   |   |   |--- Feature1 >  2.65
|   |   |   |   |--- class: 0
|   |--- Feature2 >  -0.20
|   |   |--- Feature1 <= 0.87
|   |   |   |--- Feature4 <= 0.03
|   |   |   |   |--- class: 1
|   | 

Sanity check — confirm `baseline_model` exists and shows its settings.

In [37]:
baseline_model

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at 

## Step 6 (continued) — Score the baseline on the test set

`.predict(X_test)` guesses on the 30 held-out students, blind to
`y_test`. `accuracy_score` grades those guesses — this becomes
`baseline_accuracy`, the number Forward and Backward Selection get
compared against next.

In [38]:
y_pred = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred)
baseline_accuracy


0.7333333333333333

## Step 7 — Forward Selection

### Why this step exists, when we already have `baseline_accuracy`

`baseline_accuracy = 0.733` came from training on **all 5 features**,
noise included (`Feature3`/`4`/`5` were built as pure random numbers,
with zero real connection to `Target`). That number only tells us "the
score if you throw everything at the model." It doesn't tell us *which*
of the 5 features are actually pulling their weight.

Forward Selection answers a different question: **can fewer,
better-chosen features match or beat that baseline — and which ones
actually matter?** `20_Feature_Selection_techniques.ipynb` covered this
on `diabetes.csv`; `23_Simple_Dataset_Practice.ipynb` never touched it
at all, because that notebook only ever had **one** feature — there was
nothing to select *from*. Feature selection only becomes a real
question once there's more than one candidate and some of them might be
useless, exactly like the 5 here.

### `SFS(...)`, parameter by parameter

```python
sfs_forward = SFS(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    k_features=(1, 5),
    forward=True,
    floating=False,
    scoring="accuracy",
    cv=5,
)
```

This line **builds** the search object — nothing runs yet, same as
`DecisionTreeClassifier(max_depth=4)` alone didn't train anything until
`.fit()` was called on it.

- **First argument, `DecisionTreeClassifier(max_depth=4, random_state=42)`**
  — the "judge." `SFS` makes a fresh copy of this exact model for every
  candidate feature combination it tries, trains that copy, and reads
  off its score. Same judge every time, so scores stay comparable.
- **`k_features=(1, 5)`** — a *range*, not one fixed number. Keep going
  until every size from 1 feature up to all 5 has been tried, recording
  the best score at each size.
- **`forward=True`** — start from 0 features, add the single best one,
  then keep adding one more each round. (Step 8 flips this to
  `forward=False`: start from all 5, remove one at a time.)
- **`floating=False`** — the stricter setting. If `True`, the search
  could occasionally *remove* a feature it added earlier, if a later
  addition makes that first choice look bad in hindsight. `False` means
  once a feature is added, it's locked in for good.
- **`scoring="accuracy"`** — same metric as `accuracy_score` from
  notebook 23, just computed automatically per candidate.
- **`cv=5`** — every candidate gets 5-fold cross-validated instead of
  judged on one single train/check split: cut the training rows into 5
  chunks, train on 4, check on the 5th, five times over, average the 5
  scores.

### The actual search space this will run

```mermaid
flowchart TD
    START["120 training rows\n5 candidate features:\nFeature1 .. Feature5"] --> R1
    subgraph R1 ["Round 1 - try every single feature alone (5 candidates)"]
        R1A["Feature1 alone"]
        R1B["Feature2 alone"]
        R1C["Feature3 alone"]
        R1D["Feature4 alone"]
        R1E["Feature5 alone"]
    end
    R1 --> W1["Keep whichever scored\nbest - added to the set"]
    W1 --> R2
    subgraph R2 ["Round 2 - add 1 more, try all remaining (4 candidates)"]
        R2A["+ remaining feature A"]
        R2B["+ remaining feature B"]
        R2C["+ remaining feature C"]
        R2D["+ remaining feature D"]
    end
    R2 --> W2["Keep the best pair"]
    W2 --> DOTS["Round 3 (3 candidates) ...\nRound 4 (2 candidates) ...\nRound 5 (1 candidate: all 5 left)"]
    DOTS --> DONE["k_features=(1,5): every round's\nbest score gets recorded,\nbest size overall wins"]
    style START fill:#8a8f98,color:#fff
    style W1 fill:#2a78d6,color:#fff
    style W2 fill:#2a78d6,color:#fff
    style DONE fill:#2f9e6f,color:#fff
```

Round 1 tries 5 candidates, Round 2 tries 4, Round 3 tries 3, Round 4
tries 2, Round 5 tries 1 — `5+4+3+2+1 = 15` feature combinations total.
Each one gets 5-fold cross-validated (`cv=5`), so `.fit()` is really
training `15 x 5 = 75` individual trees behind the scenes, all within
the one line below.

In [39]:
sfs_forward = SFS(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    k_features=(1, 5),
    forward=True,
    floating=False,
    scoring="accuracy",
    cv=5,
)
sfs_forward.fit(X_train, y_train)


,estimator,DecisionTreeC...ndom_state=42)
,k_features,"(1, ...)"
,scoring,'accuracy'
,forward,True
,floating,False
,verbose,0
,cv,5
,n_jobs,1
,pre_dispatch,'2*n_jobs'
,clone_estimator,True
,fixed_features,None


Check which features `sfs_forward` actually kept.

In [40]:
sfs_forward.k_feature_names_


('Feature1', 'Feature2')

Turn `sfs_forward.subsets_` into a readable table — one row per round,
tracing the score as features get added.

In [41]:
forward_table = pd.DataFrame(sfs_forward.subsets_).T
forward_table


,feature_idx,cv_scores,avg_score,feature_names
1,"(0,)","[0.7083333333333334, 0.625, 0.75, 0.7083333333...",0.725,"(Feature1,)"
2,"(0, 1)","[0.875, 0.875, 0.8333333333333334, 0.875, 0.79...",0.85,"(Feature1, Feature2)"
3,"(0, 1, 4)","[0.8333333333333334, 0.875, 0.8333333333333334...",0.841667,"(Feature1, Feature2, Feature5)"
4,"(0, 1, 2, 4)","[0.75, 0.8333333333333334, 0.8333333333333334,...",0.808333,"(Feature1, Feature2, Feature3, Feature5)"
5,"(0, 1, 2, 3, 4)","[0.6666666666666666, 0.8333333333333334, 0.875...",0.791667,"(Feature1, Feature2, Feature3, Feature4, Featu..."


The score attached specifically to the winning set (`Feature1`,
`Feature2`) — the peak row of `forward_table`.

In [43]:
sfs_forward.k_score_


np.float64(0.85)

In [44]:
forward_features = list(sfs_forward.k_feature_names_)

final_forward_model = DecisionTreeClassifier(max_depth=4, random_state=42)
final_forward_model.fit(X_train[forward_features], y_train)

forward_test_accuracy = accuracy_score(
    y_test, final_forward_model.predict(X_test[forward_features])
)
forward_test_accuracy


0.7333333333333333

### Why `forward_test_accuracy` exactly ties `baseline_accuracy`

Checked by hand: `baseline_model` (5 features) and `final_forward_model`
(2 features) predict the **exact same class for every one of the 30
test students** — not just the same count of correct guesses, literally
identical guesses, row by row. Even though `baseline_model`'s full tree
does use `Feature4`/`Feature5` somewhere in its deeper branches, none of
these 30 test students happened to land in a spot where that extra
branching actually changed their final answer.

That reframes the result: Forward Selection didn't fail to "beat" the
baseline — it **matched it exactly** while throwing away 3 of the 5
features, and *proved* (not guessed) that `Feature3`/`4`/`5` are dead
weight for these predictions. Beating a baseline built on real signal
plus useless noise was never the likely outcome — the best realistic
case is a tie like this; a worse case (like `20_Feature_Selection_techniques.ipynb`'s
diabetes result, where the smaller model scored *lower*) is also normal,
for the same "small test set is noisy" reason covered there.

## Step 8 — Backward Elimination

The mirror image of Step 7. Instead of starting empty and adding the
best feature each round, it starts with **all 5** features and removes
the *least* useful one each round, until only 1 is left. Same `SFS`
class, same judge, same `cv=5` scoring — only one setting flips:

```python
sfs_backward = SFS(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    k_features=(1, 5),
    forward=False,
    floating=False,
    scoring="accuracy",
    cv=5,
)
sfs_backward.fit(X_train, y_train)
```

`forward=False` is the only change from Step 7 — everything else means
exactly what it did there.

### The search space this runs — reversed direction

```mermaid
flowchart TD
    START["120 training rows\nall 5 features:\nFeature1 .. Feature5"] --> R1
    subgraph R1 ["Round 1 - try removing each feature, one at a time (5 candidates)"]
        R1A["Remove Feature1, keep other 4"]
        R1B["Remove Feature2, keep other 4"]
        R1C["Remove Feature3, keep other 4"]
        R1D["Remove Feature4, keep other 4"]
        R1E["Remove Feature5, keep other 4"]
    end
    R1 --> W1["Keep whichever removal\nhurt least - that feature\nstays gone for good"]
    W1 --> R2
    subgraph R2 ["Round 2 - try removing each remaining feature (4 candidates)"]
        R2A["Remove candidate A"]
        R2B["Remove candidate B"]
        R2C["Remove candidate C"]
        R2D["Remove candidate D"]
    end
    R2 --> W2["Keep the best 3-feature set"]
    W2 --> DOTS["Round 3 (3 candidates) ...\nRound 4 (2 candidates) ...\nRound 5 (1 candidate left)"]
    DOTS --> DONE["k_features=(1,5): every round's\nbest score gets recorded,\nbest size overall wins"]
    style START fill:#8a8f98,color:#fff
    style W1 fill:#eb6834,color:#fff
    style W2 fill:#eb6834,color:#fff
    style DONE fill:#2f9e6f,color:#fff
```

Same total: `5+4+3+2+1 = 15` combinations, `15 x 5 = 75` trees trained —
just approached from the opposite end. Given only 5 candidate features
here, predict before running: do you expect Forward and Backward to
land on the same winning set (`Feature1`, `Feature2`) this time, same
as `20_Feature_Selection_techniques.ipynb`'s diabetes result?

In [45]:
sfs_backward = SFS(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    k_features=(1, 5),
    forward=False,
    floating=False,
    scoring="accuracy",
    cv=5,
)
sfs_backward.fit(X_train, y_train)

,estimator,DecisionTreeC...ndom_state=42)
,k_features,"(1, ...)"
,forward,False
,scoring,'accuracy'
,floating,False
,verbose,0
,cv,5
,n_jobs,1
,pre_dispatch,'2*n_jobs'
,clone_estimator,True
,fixed_features,None


Same check as Step 7 — which features did `sfs_backward` keep this
time?

In [46]:
sfs_backward.k_feature_names_

('Feature1', 'Feature2')

Same table shape as `forward_table`, but read top (`5` features) to
bottom (`1` feature) — a removal trace instead of an addition trace.

In [47]:
backward_table = pd.DataFrame(sfs_backward.subsets_).T
backward_table


,feature_idx,cv_scores,avg_score,feature_names
5,"(0, 1, 2, 3, 4)","[0.6666666666666666, 0.8333333333333334, 0.875...",0.791667,"(Feature1, Feature2, Feature3, Feature4, Featu..."
4,"(0, 1, 2, 4)","[0.75, 0.8333333333333334, 0.8333333333333334,...",0.808333,"(Feature1, Feature2, Feature3, Feature5)"
3,"(0, 1, 4)","[0.8333333333333334, 0.875, 0.8333333333333334...",0.841667,"(Feature1, Feature2, Feature5)"
2,"(0, 1)","[0.875, 0.875, 0.8333333333333334, 0.875, 0.79...",0.85,"(Feature1, Feature2)"
1,"(0,)","[0.7083333333333334, 0.625, 0.75, 0.7083333333...",0.725,"(Feature1,)"


The score attached to `sfs_backward`'s winning set.

In [48]:
sfs_backward.k_score_


np.float64(0.85)

## Step 8 (continued) — Build the final backward-selected model, check
it on the test set

Same pattern as Step 7's final check: train a fresh model on just the
winning features, test it on the same 30 held-out rows.

In [49]:
backward_features = list(sfs_backward.k_feature_names_)

final_backward_model = DecisionTreeClassifier(max_depth=4, random_state=42)
final_backward_model.fit(X_train[backward_features], y_train)

backward_test_accuracy = accuracy_score(
    y_test, final_backward_model.predict(X_test[backward_features])
)
backward_test_accuracy


0.7333333333333333

## Step 9 — Compare all three models

Same structure as notebook 20's final table: features used vs test
accuracy, side by side, for the baseline, Forward Selection, and
Backward Elimination.

In [50]:
comparison = pd.DataFrame({
    "features_used": [X.shape[1], len(forward_features), len(backward_features)],
    "test_accuracy": [baseline_accuracy, forward_test_accuracy, backward_test_accuracy],
}, index=["All features (baseline)", "Forward Selection", "Backward Elimination"])
comparison


,features_used,test_accuracy
All features (baseline),5,0.733333
Forward Selection,2,0.733333
Backward Elimination,2,0.733333


## Key takeaways

- **`5` features → `0.733`. `2` features (either direction) → `0.733`.**
  A clean tie, not a win or a loss — and proven by hand earlier to be
  an *exact* tie: every one of the 30 test students got the identical
  prediction whether the model saw 5 features or just 2.
- Forward and Backward Selection are the **same search idea** as the
  single-threshold search in `23_Simple_Dataset_Practice.ipynb` — "try
  every option, score it, keep the best" — just applied one level up:
  instead of searching over cutoff *lines* for one feature, this
  searches over *combinations of features*, training a full model per
  candidate.
- Forward starts at 0 features and adds; Backward starts at all 5 and
  removes. Here, with only 5 candidates, **both converged on the exact
  same answer** (`Feature1`, `Feature2`) — strong evidence that pair is
  genuinely the real signal, not a fluke of which direction was
  searched. This isn't guaranteed with more features; the two can
  disagree.
- The real result of this notebook: the search **correctly identified
  the 2 real features and discarded the 3 pure-noise ones**, without
  ever being told which was which — matching accuracy while using 60%
  fewer features. That's a genuinely good outcome, distinct from
  `20_Feature_Selection_techniques.ipynb`'s diabetes result, where the
  selected model scored *lower* on the test set than the baseline — a
  reminder that a better cross-validated *training* score doesn't
  guarantee a better single-split *test* score, since a small, fixed
  test set carries its own noise.

That closes the loop on both `22_Forward_Backward_Practice.ipynb` (this
notebook, wrapper methods on 5 features) and its companion
`23_Simple_Dataset_Practice.ipynb` (the fundamentals, on 1 feature) —
together covering the full path from "what does `.fit()` actually
compute" up to "how do you decide which features to keep at all."

## Recap — the whole flow, Steps 1–6

```mermaid
flowchart TD
    A["Block A\ndataset\n150 rows, 6 columns"] --> B["Block B\nX = dataset.drop(columns=['Target'])"]
    A --> C["Block C\ny = dataset['Target']"]
    B --> D["Block D\ntrain_test_split(X, y, test_size=0.2,\nrandom_state=42, stratify=y)"]
    C --> D
    D --> E["Block E\nX_train (120 rows)"]
    D --> F["Block F\nX_test (30 rows)"]
    D --> G["Block G\ny_train (120 rows)"]
    D --> H["Block H\ny_test (30 rows)"]
    E --> I["Block I\nbaseline_model.fit(X_train, y_train)"]
    G --> I
    I --> J["Block J\ntrained baseline_model"]
    J --> K["Block K\nbaseline_model.predict(X_test)"]
    F --> K
    K --> L["Block L\ny_pred (30 predictions)"]
    L --> M["Block M\naccuracy_score(y_test, y_pred)"]
    H --> M
    M --> N["Block N\nbaseline_accuracy = 0.733"]
    style A fill:#8a8f98,color:#fff
    style D fill:#2a78d6,color:#fff
    style I fill:#2a78d6,color:#fff
    style K fill:#eb6834,color:#fff
    style N fill:#2f9e6f,color:#fff
```

### Why each block exists

**Block A — `dataset`.** The one shared table, 150 rows × 6 columns.
Purpose: everything below is only ever a slice of this same table, so a
row can never end up mismatched between `X`/`y` or train/test.

**Block B — `X`.** Drops `Target`, keeps the 5 feature columns.
Purpose: this is *only* what the model is allowed to see. If `Target`
stayed in, the model could read the answer straight off the input
instead of learning a real pattern — the score would mean nothing.

**Block C — `y`.** Just the `Target` column, kept separate.
Purpose: the answer key. Needed twice later — to teach the model
(Block G) and to grade it (Block H) — but never as something the model
gets to look at directly.

**Block D — `train_test_split`.** Shuffles `X` and `y` together (same
order for both, so row 37 of `X` always stays paired with row 37 of
`y`), then cuts the shuffle into two pools. Purpose: one pool to learn
from, a second untouched pool to prove the learning actually
generalizes.

**Block E — `X_train`, 120 rows.** The 80% share. Purpose: more
training rows generally means the model finds sturdier, less accidental
patterns.

**Block F — `X_test`, 30 rows.** The remaining 20%, held back entirely
during training. Purpose: the model's one honest shot at "new" data —
the only real proof it learned something instead of memorizing.

**Block G — `y_train`, 120 rows.** The true answers for the training
rows. Purpose: `.fit()` learns by comparing guesses against these real
answers — no true answers, no learning signal at all.

**Block H — `y_test`, 30 rows.** The true answers for the test rows —
notice nothing connects `H` straight to `K`. Purpose: kept apart from
prediction on purpose, so it can grade fairly afterward instead of
leaking the answer into the guess.

**Block I — `baseline_model.fit(...)`.** Purpose: the actual training —
the model looks at all 120 rows' features *and* their true answers
together, and adjusts its internal yes/no questions to match as many as
it can.

**Block J — trained `baseline_model`.** Purpose: the finished rulebook
`.fit()` produced. Nothing is predicted yet here — just the learned
rules, sitting in memory, ready to be used.

**Block K — `.predict(X_test)`.** Purpose: run the 30 test rows'
features (features only) through that rulebook and get a guess per row.
Only `X_test` goes in, never `y_test` — this copies the real world,
where you'd only ever have features and need the model to guess the
unknown answer.

**Block L — `y_pred`, 30 guesses.** Purpose: the model's answers, made
with zero knowledge of the real ones.

**Block M — `accuracy_score(y_test, y_pred)`.** Purpose: the *only*
place `y_test` finally gets used — comparing guesses to truth, row by
row, and turning "how many matched" into one fraction.

**Block N — `baseline_accuracy = 0.733`.** Purpose: one number
summarizing the entire pipeline above it — 73.3% of the 30 test guesses
matched the truth.

### The same flow, as a story

Picture the 150 rows as 150 flashcards. Each card has 5 clues on the
front and one hidden answer on the back.

**Block A** is the full stack of 150 cards, answers still attached.
Before doing anything else, we peel the answer off every card: the
clue-only cards become **Block B**, and the pile of peeled-off answers
becomes **Block C** — kept safe, not thrown away.

Now we shuffle the whole deck once, the same way for the cards and
their matching answers, and deal it into two piles (**Block D**). The
bigger pile — 120 cards — becomes the **study pile** (**Block E**), with
its matching answers (**Block G**) handed over too. The smaller pile —
30 cards — becomes the **quiz pile** (**Block F**), and its answers
(**Block H**) get locked away for now.

**Block I** is study time: the learner sees all 120 clue cards *and*
their real answers side by side, and keeps adjusting its own guessing
rules until it matches as many as it can. What it ends up with is
**Block J** — its own personal rulebook, nothing more.

**Block K** is quiz time: the learner is handed the 30 quiz cards —
clues only, answers still hidden — and has to guess using nothing but
that rulebook. Its 30 guesses become **Block L**, made with zero
peeking.

Finally, **Block M** is grading time: the locked-away answers from
**Block H** get revealed, compared against the 30 guesses one by one,
and the result becomes **Block N** — 73.3%, or about 22 out of 30
correct, on a quiz the learner had genuinely never seen before.
